In [1]:
import os
from datetime import datetime
import json

In [2]:
class Colour:
    BLUE = '\033[94m'
    CYAN = '\033[96m'
    GREEN = '\033[92m'
    YELLOW = '\033[93m'
    RED = '\033[91m'
    BOLD = '\033[1m'
    END = '\033[0m'

In [3]:
def clean_number(value):
    if value is None:
        return 0.0
    
    # if already a numer
    if isinstance(value, (int, float)):
        return float(value)
    else:
        clean_str = str(value).strip()
        is_negative = clean_str.endswith('-') or clean_str.startswith('-')
        
        clean_str = clean_str.replace(',', '.') # french comma -> point
        clean_str = ''.join(c for c in clean_str if c.isdigit() or c == '.' or c == '-')
        
        try:
            val = float(clean_str)
            return val if not is_negative else -abs(val)
        except ValueError:
            return 0.0

In [4]:
def parse_date(date_str):
    if not date_str:
        return None
    try:
        return datetime.strptime(date_str, "%Y-%m-%d")
    except ValueError:
        return None

In [5]:
def verification_contrat(chemin_fichier):
    if not os.path.exists(chemin_fichier):
        print(f"{Colour.RED}❌ Fichier introuvable : {chemin_fichier}{Colour.END}")
        return

    try:
        with open(chemin_fichier, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except json.JSONDecodeError:
        print(f"{Colour.RED}❌ JSON invalide.{Colour.END}")
        return

    # format (TEST)
    if "type_contrat" not in data and "parties_prenantes" not in data:
         if "partie_A" not in data:
            print(f"{Colour.YELLOW}⚠️ Ce fichier ne ressemble pas au format de contrat amélioré.{Colour.END}")
            return

    titre = data.get('type_contrat', 'Contrat Inconnu')
    objet = data.get('objet_contrat', 'Non spécifié')
    print(f"{Colour.BLUE}{Colour.BOLD}" + "="*60)
    print(f" AUDIT CONTRAT : {titre}")
    print(f" Objet : {objet}")
    print("="*60 + f"{Colour.END}")

    errors = []
    warnings = []

    print(f"{Colour.YELLOW}" + "-" * 70 + f"{Colour.END}")
    print(f"{Colour.CYAN}analyse des parties{Colour.END}")
    
    parties = []
    for key in ['partie_A', 'partie_B']:
        p = data.get(key)
        if p:
            parties.append(p)
            nom = p.get('nom', 'Inconnu')
            role = p.get('role', '?')
            identifiant = p.get('identifiant')
            adresse = p.get('adresse')
            
            status = f"{Colour.GREEN}OK{Colour.END}"
            details = []
            
            if not identifiant:
                details.append("Pas de SIRET/ID")
                warnings.append(f"Partie '{nom}' : Identifiant légal manquant.")
                status = f"{Colour.YELLOW}ATTENTION{Colour.END}"
            
            if not adresse:
                details.append("Pas d'adresse")
                status = f"{Colour.YELLOW}INCOMPLET{Colour.END}"

            print(f"👤 {role} : {nom} -> {status} ({', '.join(details) if details else 'Complet'})")
    
    if len(parties) < 2:
        errors.append("Le contrat semble n'avoir qu'une seule partie identifiée (ou aucune).")

    # temporal analysis
    print(f"{Colour.YELLOW}" + "-" * 70 + f"{Colour.END}")
    print(f"{Colour.CYAN}analyse temporelle{Colour.END}")
    
    d_sign = parse_date(data.get('date_signature'))
    d_debut = parse_date(data.get('date_debut_effet'))
    d_fin = parse_date(data.get('date_fin_prevue'))
    
    if d_sign:
        print(f"📅 Date Signature : {d_sign.strftime('%d/%m/%Y')}")
    else:
        errors.append("Date de signature manquante ou illisible.")

    if d_debut:
        print(f"🚀 Date Début     : {d_debut.strftime('%d/%m/%Y')}")
    
    if d_fin:
        print(f"🏁 Date Fin       : {d_fin.strftime('%d/%m/%Y')}")

    if d_sign and d_debut:
        if d_sign > d_debut:
            warnings.append(f"Antédatation suspecte : Le contrat a été signé {(d_sign - d_debut).days} jours APRÈS sa date de début d'effet.")
        elif d_sign > datetime.now():
             errors.append(f"Signature dans le futur ! ({d_sign.strftime('%d/%m/%Y')})")

    if d_debut and d_fin:
        # check negative duraitno
        if d_debut > d_fin:
            errors.append(f"Incohérence majeure : La date de fin est AVANT la date de début.")
        
        # compute duration
        duree_jours = (d_fin - d_debut).days
        print(f"⏱️  Durée calculée : {duree_jours} jours ({duree_jours/365:.1f} ans)")

    if d_sign and d_fin:
        if d_sign > d_fin:
            errors.append("Contrat signé APRÈS sa date de fin prévue (Absurde).")

    # small financial analysis
    print(f"{Colour.YELLOW}" + "-" * 70 + f"{Colour.END}")
    print(f"{Colour.CYAN}analyse financière{Colour.END}")
    montant = data.get('montant_principal')
    devise = data.get('devise', 'EUR')
    freq = data.get('frequence_paiement', 'Non spécifiée')
    
    if montant:
        val_montant = clean_number(montant)
        if val_montant <= 0:
            warnings.append(f"Montant du contrat nul ou négatif : {val_montant}")
        else:
            print(f"💰 Montant facial : {val_montant:,.2f} {devise} ({freq})")
            
            # alert
            if devise not in ['EUR', 'USD', 'GBP'] and 'France' in str(data):
                warnings.append(f"Devise inhabituelle pour un contexte français : {devise}")
    else:
        warnings.append("Aucun montant principal détecté dans le contrat.")


    # final report
    print(f"{Colour.YELLOW}" + "-" * 70 + f"{Colour.END}")
    if errors:
        print(f"{Colour.RED}🚨 ERREURS CRITIQUES DÉTECTÉES :{Colour.END}")
        for e in errors:
            print(f"  ❌ {e}")
    elif warnings:
        print(f"{Colour.YELLOW}⚠️ POINTS DE VIGILANCE :{Colour.END}")
        for w in warnings:
            print(f"  🔸 {w}")
    else:
        print(f"{Colour.GREEN}✅ Le contrat semble cohérent sur la forme et les dates.{Colour.END}")
    
    print(f"{Colour.YELLOW}" + "-" * 70 + f"{Colour.END}")

In [6]:
verification_contrat("extracted/customer_contract2.json")

⚠️ Ce fichier ne ressemble pas au format de contrat amélioré.
